# RoadShield Vision V2 — Full RDD2022 (six countries)

**Start with the data-only cells (1–4).** Tomorrow choose a free T4 GPU in Colab and run cell 5. The official Figshare archive contains all six countries and seven capture subsets, including China Drone and MotorBike. Only labelled images enter the supervised training split. Save the complete archive and checkpoints in Google Drive. The full archive is large (~13 GB); make sure Drive has space for it and model checkpoints, or choose temporary local storage below. No paid compute is required or enabled here.

Source: https://doi.org/10.6084/m9.figshare.21431547.v1 . This notebook does not claim that real training or download has already happened. It uses the pinned RoadShield training code, not the previous India-only checkpoint. Data split is grouped by perceptual hash within each country; it is not a geographically independent or Egyptian test. Do not deploy an unvalidated model.

In [ ]:
# 1 — Mount Drive and pin the training code (CPU is fine for data prep).
from google.colab import drive
from pathlib import Path
import os, subprocess, shutil, sys
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/RoadShield/RDD2022_full')
ROOT.mkdir(parents=True, exist_ok=True)
SAVE_ARCHIVE_TO_DRIVE = True # False = temporary archive, lost when Colab resets
CACHE = ROOT / 'source' if SAVE_ARCHIVE_TO_DRIVE else Path('/content/rdd2022_source')
CACHE.mkdir(parents=True, exist_ok=True)
RAW = Path('/content/rdd2022_all_raw')
PREP = ROOT / 'prepared'
OUT = ROOT / 'vision_v2_24epochs'
for p in (RAW, PREP, OUT):
    p.mkdir(parents=True, exist_ok=True)
REPO = Path('/content/roadshield-v2-code')
REV = '1019268ee96fa6f0c1dd33d29ea3c716e7f41725'
if not REPO.exists():
    subprocess.run(['git','init',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--depth=1','https://github.com/Alaa2134/roadshield-ai.git',REV],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',REV],check=True)
print('Archive location:', CACHE)
print('Free local disk GiB:', round(shutil.disk_usage('/content').free / 2**30, 1))
print('Check available Google Drive quota separately if saving there.')

In [ ]:
# 2 — Official source, verified full archive, resumable HTTP download.
import requests, json, hashlib, time
from datetime import datetime, timezone
API = 'https://api.figshare.com/v2/articles/21431547'
response = requests.get(API, timeout=60)
response.raise_for_status()
meta = response.json()
licence = (meta.get('license') or {}).get('name','UNKNOWN')
assert licence in {'CC BY 4.0','CC BY-SA 4.0','CC0','CC BY','CC BY-SA'}, f'Review licence: {licence}'
files = [f for f in meta['files'] if f['name'].lower().endswith('.zip')]
print([(f['name'], f['size']) for f in files])
# The official release is one outer archive containing country ZIPs.
f = next((f for f in files if f['name'].lower() == 'rdd2022.zip'), None)
if f is None:
    assert len(files)==1, 'Official file list changed: inspect above before continuing'
    f=files[0]
expected = int(f['size'])
md5 = f.get('computed_md5') or f.get('supplied_md5')
assert md5, 'Official MD5 not available; stop and review'
archive = CACHE / Path(f['name']).name
partial = archive.with_suffix('.zip.part')
def digest(path):
    a,b = hashlib.md5(),hashlib.sha256()
    with path.open('rb') as src:
        for chunk in iter(lambda:src.read(8*1024*1024),b''):
            a.update(chunk); b.update(chunk)
    return a.hexdigest(), b.hexdigest()
if archive.exists():
    assert archive.stat().st_size==expected, 'Existing ZIP size incorrect; move it out before retry'
    assert digest(archive)[0].lower()==md5.lower(), 'Existing ZIP checksum incorrect; move it out before retry'
else:
    print(f'Downloading {expected/1e9:.2f} GB; saved partial bytes are reused on retry.')
    for attempt in range(12):
        have = partial.stat().st_size if partial.exists() else 0
        assert have<=expected, 'Partial file exceeds expected size'
        if have==expected: break
        try:
            headers={'Range':f'bytes={have}-'} if have else {}
            with requests.get(f['download_url'],headers=headers,stream=True,timeout=(30,90)) as r:
                r.raise_for_status()
                if have:
                    assert r.status_code==206 and r.headers.get('Content-Range','').startswith(f'bytes {have}-'), 'Server did not honor resume; preserve partial file and retry later'
                else:
                    assert r.status_code==200, f'Unexpected status: {r.status_code}'
                with partial.open('ab' if have else 'wb') as dst:
                    next_report=have+100*1024*1024
                    for chunk in r.iter_content(4*1024*1024):
                        if not chunk: continue
                        dst.write(chunk); have+=len(chunk)
                        if have>=next_report:
                            print(f'{have/1e9:.2f}/{expected/1e9:.2f} GB ({100*have/expected:.1f}%)',flush=True)
                            next_report=have+100*1024*1024
        except requests.RequestException as e:
            print('Connection interrupted:', type(e).__name__, 'retrying from disk')
            time.sleep(min(60,2**attempt))
    assert partial.exists() and partial.stat().st_size==expected, 'Incomplete download; rerun this cell'
    assert digest(partial)[0].lower()==md5.lower(), 'MD5 mismatch; do not train'
    partial.replace(archive)
actual_md5, sha256 = digest(archive)
assert actual_md5.lower()==md5.lower()
record={'source':API,'doi':meta.get('doi'),'licence':licence,'citation':meta.get('citation'),'file':f['name'],'bytes':expected,'md5':actual_md5,'sha256':sha256,'verified_at':datetime.now(timezone.utc).isoformat()}
(ROOT/'LICENSE_RECORD.json').write_text(json.dumps(record,indent=2))
print('Official archive verified:', archive)

In [ ]:
# 3 — Extract every labelled country subset locally; keep the original archive.
import zipfile, tempfile
EXPECTED = {'India','Japan','Czech','Norway','United_States','China_MotorBike','China_Drone'}
aliases = {'india':'India','japan':'Japan','czech':'Czech','czechrepublic':'Czech','norway':'Norway','unitedstates':'United_States','usa':'United_States','chinamotorbike':'China_MotorBike','chinadrone':'China_Drone'}
def canonical(s):
    return aliases.get(''.join(c for c in s.lower() if c.isalnum()))
def extract_safe(z, target):
    target = target.resolve()
    for info in z.infolist():
        p = Path(info.filename)
        if '__MACOSX' in p.parts or any(x.startswith('._') for x in p.parts): continue
        dest = (target/p).resolve()
        assert dest.is_relative_to(target), 'Unsafe archive path'
        z.extract(info,target)
with zipfile.ZipFile(archive) as outer:
    members=[i for i in outer.infolist() if i.filename.lower().endswith('.zip') and '__MACOSX' not in Path(i.filename).parts and not Path(i.filename).name.startswith('._')]
    print('Nested archives:',[i.filename for i in members])
    assert members, 'Expected nested country archives not found; inspect official archive layout'
    for member in members:
        name=canonical(Path(member.filename).stem)
        if name and (RAW/name/'.ready').is_file(): continue
        with tempfile.TemporaryDirectory(dir='/content') as tmp:
            tmp=Path(tmp)
            inner=tmp/'country.zip'
            with outer.open(member) as src, inner.open('wb') as dst:
                shutil.copyfileobj(src,dst,8*1024*1024)
            with zipfile.ZipFile(inner) as z:
                extract_safe(z,tmp/'extracted')
            inner.unlink()
            candidates=[p for p in (tmp/'extracted').rglob('xmls') if p.parent.name=='annotations' and p.parent.parent.name=='train']
            assert candidates, f'No labelled train folder in {member.filename}'
            for xmls in candidates:
                country=canonical(xmls.parents[2].name) or name
                assert country in EXPECTED, f'Unknown country folder: {xmls.parents[2].name}'
                source=xmls.parents[2]
                assert (source/'train/images').is_dir(), f'Missing images: {country}'
                target=RAW/country
                if target.exists() and (target/'.ready').exists(): continue
                assert not target.exists(), f'Incomplete extraction at {target}; inspect before retry'
                shutil.move(str(source),str(target))
                (target/'.ready').touch()
                print(country, 'label XMLs:',len(list((target/'train/annotations/xmls').glob('*.xml'))),flush=True)
assert all((RAW/c/'.ready').is_file() for c in EXPECTED), f'Missing subsets: {[c for c in EXPECTED if not (RAW/c/".ready").exists()]}'
print('All seven subsets extracted.')

In [ ]:
# 4 — Prepare deterministic labelled train/validation/test manifests.
subprocess.run([sys.executable,'-u',str(REPO/'ml/datasets/prepare_rdd2022.py'),'--raw',str(RAW),'--out',str(PREP),'--seed','42'],check=True)
manifest=json.loads((PREP/'MANIFEST.json').read_text())
assert set(manifest['per_country'])==EXPECTED, 'Not all country subsets included'
for c, v in sorted(manifest['per_country'].items()):
    assert all(v[s]['images']>0 for s in ('train','val','test')), f'Empty split: {c}'
    print(c, {s:v[s]['images'] for s in ('train','val','test')})
print('Total labelled splits:',manifest['counts'])
print('Dataset version:',manifest['version'])
print('DATA READY — saved manifests:', PREP)

## 5 — Train tomorrow (free GPU only)
Reconnect and run cells 1–4 again if the local runtime reset. The verified Drive ZIP and manifests persist; extracted data is rebuilt locally. This is a new full-dataset run, not a resume from the India-only checkpoint. The current trainer uses 24 epochs and batch 2; it saves last.pt after each completed epoch. The training script does not implement balanced sampling or AMP. Inspect per-country metrics and consider balancing in a separate experiment.

In [ ]:
# 5 — Train and save checkpoints.
import torch
assert torch.cuda.is_available(), 'Enable free GPU in Runtime settings before training'
subprocess.run(['nvidia-smi','-L'],check=True)
assert all((RAW/c/'.ready').is_file() for c in EXPECTED), 'Run data setup cells first'
assert (PREP/'MANIFEST.json').is_file()
subprocess.run([sys.executable,'-m','pip','-q','install','onnx','onnxscript'],check=True)
cmd=[sys.executable,'-u',str(REPO/'ml/detector/train_detector.py'),'--data',str(PREP),'--raw',str(RAW),'--out',str(OUT),'--epochs','24','--batch','2','--seed','42','--pretrained','--onnx']
if (OUT/'last.pt').exists(): cmd.append('--resume')
print('Resuming full dataset' if '--resume' in cmd else 'Starting full dataset training',flush=True)
subprocess.run(cmd,check=True)

In [ ]:
# 6 — Report real test metrics; candidate only, no deployment.
m=json.loads((OUT/'metrics.json').read_text())
assert not m['smoke_test']
print(json.dumps(m,indent=2))
print('Artifacts:',[p.name for p in OUT.iterdir()])
print('Egyptian pilot test still required.')